# faststochtree GPU reprexes — the CUDA side

These two benchmarks are the CUDA counterparts of the Metal reprexes in
`experimental/gpu/metal/`. Together they show *why* a fully GPU-resident BART
sampler is viable on CUDA but walled on Apple Metal:

| Claim | Metal reprex | CUDA reprex (here) |
| --- | --- | --- |
| **Launch overhead** is high and un-amortizable on Metal | `dispatch_overhead` | **1. launch_overhead** — naive launches vs CUDA Graph replay |
| **Device-wide barrier**: Metal has none, its software barrier cliffs | `grid_barrier` | **2. grid_barrier** — hardware `grid.sync()`, swept over block count |

### Why these two constants matter
The BART backfit must synchronize across all observations to update the residual
**at least once per tree** (the initial Metal experiments used several device-wide
syncs per tree). So a barrier — and the phase launch around it — recurs many times
per sweep, every sweep of a long run. The *per-barrier* and *per-launch* constant is
therefore what decides feasibility, not any single run's total.

### How to run
Open in Google Colab, then **Runtime → Change runtime type → GPU** (a free T4 is
plenty). Run the cells top to bottom: each `%%writefile` cell drops a `.cu` file and
the next cell compiles and runs it with `nvcc`. Measured numbers below are from a
free Colab T4 (40 SMs).

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv && nvcc --version | tail -1

---
## 1. Launch overhead — and how CUDA Graphs erase it

Counterpart to Metal `dispatch_overhead`. Launch an empty kernel `N` times, two ways:
**naive** (`N` separate `<<<>>>` launches) and **graph** (capture the `N` launches
once, replay with a single `cudaGraphLaunch`).

Metal's per-command-buffer cost is ~25 µs, and it has **no graph replay** that can
cross a device-wide sync. CUDA naive launches are already cheaper, and graph replay
drops them further still.

Measured (Colab T4):
```
naive 5000 launches : 10.573 us/launch
graph replay      :  1.606 us/launch  (6.6x cheaper)
```
Both beat Metal's ~25 µs/command buffer, and graph replay (~1.6 µs) does so by ~15x —
while Metal has no replay mechanism that can span a device-wide sync at all.

In [ ]:
%%writefile launch_overhead.cu
// Empty-kernel launch cost: naive per-launch vs CUDA Graph replay.
#include <cuda_runtime.h>
#include <cstdio>
#define CK(c) do{ cudaError_t e=(c); if(e!=cudaSuccess) \
  printf("CUDA error %s:%d %s\n",__FILE__,__LINE__,cudaGetErrorString(e)); }while(0)

__global__ void noop() {}

int main() {
    cudaDeviceProp p; CK(cudaGetDeviceProperties(&p, 0));
    printf("device: %s\n", p.name);
    const int N = 5000;
    cudaStream_t s; CK(cudaStreamCreate(&s));
    cudaEvent_t a, b; cudaEventCreate(&a); cudaEventCreate(&b);

    // naive: N separate launches.
    noop<<<1,1,0,s>>>(); CK(cudaStreamSynchronize(s));            // warmup
    cudaEventRecord(a, s);
    for (int i = 0; i < N; ++i) noop<<<1,1,0,s>>>();
    cudaEventRecord(b, s); CK(cudaStreamSynchronize(s));
    float naive_ms = 0; cudaEventElapsedTime(&naive_ms, a, b);

    // graph: capture N launches, replay once.
    cudaGraph_t graph; cudaGraphExec_t exec;
    CK(cudaStreamBeginCapture(s, cudaStreamCaptureModeGlobal));
    for (int i = 0; i < N; ++i) noop<<<1,1,0,s>>>();
    CK(cudaStreamEndCapture(s, &graph));
    CK(cudaGraphInstantiate(&exec, graph, 0));
    CK(cudaGraphLaunch(exec, s)); CK(cudaStreamSynchronize(s));   // warmup
    cudaEventRecord(a, s);
    CK(cudaGraphLaunch(exec, s));
    cudaEventRecord(b, s); CK(cudaStreamSynchronize(s));
    float graph_ms = 0; cudaEventElapsedTime(&graph_ms, a, b);

    printf("naive %d launches : %.3f us/launch\n", N, naive_ms * 1e3 / N);
    printf("graph replay      : %.3f us/launch  (%.1fx cheaper)\n",
           graph_ms * 1e3 / N, naive_ms / graph_ms);
    return 0;
}


In [ ]:
!nvcc -O3 -std=c++17 -arch=native launch_overhead.cu -o launch_overhead && ./launch_overhead

---
## 2. Device-wide barrier — `grid.sync()`

Counterpart to Metal `grid_barrier`, and the decisive one. A single kernel does a
trivial bit of work and then a device-wide barrier, repeated many times, launched
**cooperatively** (which guarantees all blocks are co-resident). We sweep the block
count and report ns/barrier.

Measured (Colab T4, 40 SMs, max 160 co-resident blocks):
```
  blocks   ns/barrier
       1       1678.0
      40       1856.8
      80       1963.2
     160       2142.2
```
`grid.sync()` stays **~1.7–2.1 µs/barrier and essentially flat** — it drifts only 1.3x
(1678 → 2142 ns) across a 160x increase in blocks. Contrast the Metal twin, whose
software barrier goes from ~1 µs at a few threadgroups to *tens of ms* and then
livelocks — because Metal has neither a co-residency guarantee nor a hardware grid
barrier. That difference is the whole case for a single-chain GPU-resident sampler on
CUDA. (The slight upward drift here is from more blocks contending on the single
`acc` atomic; with no contention it is flatter still.)

In [ ]:
%%writefile grid_barrier.cu
// CUDA's hardware device-wide barrier (cooperative-groups grid.sync()), swept over
// block count. The point: it stays cheap and FLAT as blocks grow -- the opposite of
// the Metal software barrier (../metal/grid_barrier), which cliffs and livelocks.
#include <cuda_runtime.h>
#include <cooperative_groups.h>
#include <cstdio>
namespace cg = cooperative_groups;
#define CK(c) do{ cudaError_t e=(c); if(e!=cudaSuccess) \
  printf("CUDA error %s:%d %s\n",__FILE__,__LINE__,cudaGetErrorString(e)); }while(0)

// n_barriers is just a repetition count so we can average the per-barrier cost --
// it is NOT a per-sweep figure (the real count is barriers-per-tree x #trees).
__global__ void barrier_bench(int n_barriers, float* acc) {
    cg::grid_group grid = cg::this_grid();
    for (int i = 0; i < n_barriers; ++i) {
        if (threadIdx.x == 0) atomicAdd(acc, 1.0f);   // trivial work between barriers
        grid.sync();                                  // <-- the hardware grid barrier
    }
}

int main() {
    const int n_barriers = 1000, threads = 256;
    cudaDeviceProp p; CK(cudaGetDeviceProperties(&p, 0));
    // Max blocks that can be co-resident (the cap for a cooperative launch).
    int per_sm = 0;
    CK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&per_sm, barrier_bench, threads, 0));
    int max_blocks = per_sm * p.multiProcessorCount;
    printf("device: %s  (%d SMs, max co-resident blocks = %d)\n\n",
           p.name, p.multiProcessorCount, max_blocks);

    float* acc; CK(cudaMalloc(&acc, sizeof(float)));
    cudaEvent_t a, b; cudaEventCreate(&a); cudaEventCreate(&b);
    printf("%8s %12s\n", "blocks", "ns/barrier");
    int pts[] = {1, max_blocks / 4, max_blocks / 2, max_blocks};
    for (int blocks : pts) {
        if (blocks < 1) continue;
        CK(cudaMemset(acc, 0, sizeof(float)));
        void* args[] = {(void*)&n_barriers, &acc};
        cudaEventRecord(a);
        CK(cudaLaunchCooperativeKernel((void*)barrier_bench, dim3(blocks), dim3(threads), args));
        cudaEventRecord(b); CK(cudaEventSynchronize(b));
        float ms = 0; cudaEventElapsedTime(&ms, a, b);
        printf("%8d %12.1f\n", blocks, ms * 1e6 / n_barriers);
    }
    return 0;
}


In [ ]:
!nvcc -O3 -std=c++17 -arch=native grid_barrier.cu -o grid_barrier && ./grid_barrier

---
## The punchline

| | Metal (M1 Max) | CUDA (Colab T4) |
| --- | --- | --- |
| per-launch / per-dispatch | ~25 µs/cmd buffer, no graph replay across a sync | naive ~10 µs, **graph replay ~1.6 µs** |
| device-wide barrier | software only; ~1 µs at a few threadgroups, **cliffs to tens of ms / livelocks** | `grid.sync()` **~1.7–2.1 µs, ~flat** (1→160 blocks) |

CUDA's **cooperative launch** guarantees all blocks are co-resident (so a barrier
can't livelock) and `grid.sync()` gives a hardware fast path; Metal offers neither.
Because the residual sync recurs at least once per tree on every sweep, that
per-barrier gap compounds — which is why the next-steps doc rates a CUDA
GPU-resident sampler **promising** and a Metal single-chain one a **non-starter**.